# The Agent Cycle: Lifecycle & Control

While reasoning (ReAct) defines how an agent thinks, the **Agent Cycle** defines how it executes. This foundational architecture manages the iterative loop of perception and action, ensuring that an agent remains stable, respects resource limits, and persists its state across multiple turns.

## 1. The ORPA Framework
Industrial-grade agents often follow the **ORPA** (Observe, Reason, Plan, Act) lifecycle:

1. **Observe**: Ingest the current state, tool outputs, and user feedback.
2. **Reason**: Analyze the observation against the original goal.
3. **Plan**: Adjust the roadmap based on new information.
4. **Act**: Execute the next specific tool or response.

This cycle ensures that the agent is not just reacting, but proactively managing a strategy.

In [ ]:
import os
import time
from typing import List, Dict, Any
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool
from langchain.agents import create_react_agent, AgentExecutor

load_dotenv()

# Central model configuration
model = "gemini-2.5-flash"
llm = ChatGoogleGenerativeAI(model=model, temperature=0)

# Shared Base Template for ReAct Agents
base_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: {input}
Thought: always start with a high-level reasoning plan
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

print(f"Agent Cycle environment initialized with model: {model}")

## 2. Manual State Tracking
A "Stateful" agent maintains a persistent record of its journey. This `AgentState` prevents redundant tool calls and allows the agent to resume if interrupted. In this manual example, the state is persisted in a local dictionary.

In [ ]:
class AgentState:
    def __init__(self, goal: str):
        self.goal = goal
        self.history: List[Dict[str, str]] = []
        self.completed_steps: List[str] = []
        self.iterations = 0

    def add_step(self, thought: str, action: str, observation: str):
        self.history.append({"thought": thought, "action": action, "observation": observation})
        self.completed_steps.append(action)
        self.iterations += 1

    def summary(self):
        print(f"--- Agent State (Iter: {self.iterations}) ---")
        for i, step in enumerate(self.history):
            print(f"Step {i+1}: {step['action']} -> Result: {step['observation']}")

state = AgentState("Find the length of 'Antigravity' and subtract 1.")
state.add_step("I need the length.", "calculate_length", "11")
state.add_step("Subtracting 1.", "math_subtract", "10")

state.summary()

## 3. Cycle Control: Iteration Limits & Timeouts
Agents can enter infinite loops if a tool output is ambiguous or if the reasoning fails to converge. Control flows like `max_iterations` and `max_execution_time` are critical for safety and cost management.

In [ ]:
@tool
def wait_tool(seconds: int) -> str:
    """Simulates a long-running process. Input should be an integer representing seconds."""
    time.sleep(int(seconds))
    return f"Finished waiting for {seconds} seconds."

tools = [wait_tool]
prompt = PromptTemplate.from_template(base_template)
agent = create_react_agent(llm, tools, prompt)

# Setting hard limits to prevent resource exhaustion
executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True, 
    max_iterations=2,           # Stop after 2 reasoning cycles
    max_execution_time=5,       # Stop after 5 seconds total
    handle_parsing_errors=True
)

print("Execution with tight control flow limits:")
try:
    executor.invoke({"input": "Use the wait tool once for 10 seconds."})
except Exception as e:
    print(f"Cycle Terminated Safely: {str(e)}")

## 4. Reflection & Progress Verification
The Cycle should include an explicit "Verification" phase. This ensures that the agent is actually making progress toward the goal, rather than repeating identical actions.

In [ ]:
verification_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:
Question: {input}
Thought: always begin by verifying your progress and then formulating the next step.
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Final Answer: the final answer to the original input question

Begin!
Question: {input}
Thought:{agent_scratchpad}"""

v_prompt = PromptTemplate.from_template(verification_template)
v_agent = create_react_agent(llm, tools, v_prompt)
v_executor = AgentExecutor(agent=v_agent, tools=tools, verbose=True, handle_parsing_errors=True)

print("Cycle with Internal Progress Verification:")
v_executor.invoke({"input": "Wait for 1 second and then finish."})

## 5. The Master Cycle Project: The Self-Monitoring Agent
This final project integrates all cycle concepts. It demonstrates an agent that manages its own state, monitors its iterations, and handles its own lifecycle transitions between reasoning and execution.

In [ ]:
@tool
def log_cycle_event(log_entry: str) -> str:
    """Logs a specific event in the cycle. Entry should be like '[PHASE] - activity_details'."""
    return f"Cycle Log recorded: {log_entry}"

@tool
def fetch_system_status() -> str:
    """Simulates fetching the current health of a system."""
    return "System: ONLINE | Load: 45% | Error_Rate: 0.02%"

cycle_tools = [log_cycle_event, fetch_system_status]

master_template = """You are a Self-Monitoring Agent. You have access to: {tools}

You must log your Cycle Phase (Observe, Reason, Plan, Act) using the log_cycle_event tool before every action.

Use the following format:
Question: {input}
Thought: reasoning about the plan
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: result
...
Final Answer: result

Begin!
Question: {input}
Thought:{agent_scratchpad}"""

m_prompt = PromptTemplate.from_template(master_template)
m_agent = create_react_agent(llm, cycle_tools, m_prompt)
m_executor = AgentExecutor(agent=m_agent, tools=cycle_tools, verbose=True, handle_parsing_errors=True)

print("\nRunning Self-Monitoring Lifecycle Cycle:")
m_executor.invoke({"input": "Check system status and log your reasoning phase."})

--- 

## Conclusion
The **Agent Cycle** is the heartbeat of any autonomous system. By defining clear stages for observation, reasoning, and action, and wrapping them in state tracking and resource guardrails, developers can build agents that are as predictable as they are capable. This completes the **01-Foundation** module.